Von Mistral generiert, austesten!

# **07_fire_non_fire_comparison.ipynb**
**Objective:** Compare woody cover trajectories between fire and non-fire pixels in the Mediterranean region, including clustering, covariate matching, and modeling.

---

## **📌 Overview**
This script performs a **complete analysis** for **RQ3**:
- **Data Preparation:** Loads Woody Cover, Burned Area, MODIS IGBP, and ecoregions.
- **Feature Extraction:** Computes trajectory features for fire and non-fire pixels.
- **Covariate Matching:** Finds matched non-fire pixels for fire pixels.
- **Clustering:** Identifies clusters of trajectories (fire + non-fire).
- **Modeling:** Explains cluster membership using Random Forest.
- **Visualization:** Plots for trajectories, cluster validation, and feature importance.
- **Export:** Saves results as raster and CSV.

---

## **🔧 Main Steps**

### **1️⃣ Setup & Load Data**
- Loads all necessary raster data (Woody Cover, Burned Area, MODIS IGBP, ecoregions).
- Filters to the **Mediterranean ecoregion**.

### **2️⃣ Pixel Categorization**
- **Fire Pixels:** Pixels with ≥1 fire.
- **Non-Fire Pixels:** Pixels without fire.
- Computes the number of pixels per category.

### **3️⃣ Feature Extraction**
- **Fire Pixels:**
  - Extracts trajectories for relative years -5 to +10.
  - Computes **8 features** (e.g., pre-fire woody cover, fire loss, recovery).
- **Non-Fire Pixels:**
  - Extracts trajectories for the last 5 years (as "pre-fire" equivalent).
  - Computes **2 features** (pre-fire woody cover, trend).

### **4️⃣ Covariate Matching**
- **Matching Criteria:** Pre-fire woody cover and trend.
- **Method:** Nearest Neighbors (1:1 matching).
- **Result:** Matched non-fire pixels for each fire pixel.

### **5️⃣ Clustering (k-Medoids)**
- **Dimensionality Reduction:** Self-Organizing Map (SOM).
- **Clustering:** k-Medoids on the SOM codebook.
- **Validation:** Silhouette Score, Calinski-Harabasz, Davies-Bouldin.
- **Recommended k-values:** Based on validation metrics.

### **6️⃣ Final Clustering**
- Runs clustering with the chosen **k**.
- **Cluster Labels:** Fire and non-fire pixels are clustered together.

### **7️⃣ Cluster Interpretation**
- **Mean Trajectories:** Plots average development per cluster.
- **Feature Profiles:** Heatmap of cluster features.
- **Fire vs. Non-Fire Comparison:** Statistical tests (Mann-Whitney) for recovery rates.

### **8️⃣ Modeling**
- **Target Variable:** Cluster membership.
- **Predictors:** Fire status, pre-fire woody cover, pre-fire trend.
- **Model:** Random Forest (feature importance).

### **9️⃣ Export Results**
- **Cluster Labels:** As raster (`cluster_labels_mediterranean.tif`).
- **Validation Metrics:** As CSV (`cluster_validation_results.csv`).
- **Visualizations:** Plots for trajectories, cluster validation, and feature importance.

---
## **📂 Output Directory**
All results are saved in `out_dir = <working_directory>/_Runs/07_Feuer_vs_NichtFeuer/`.


In [ ]:
# =============================================================================

# 07_feuer_vs_nichtfeuer_analysis.ipynb

# RQ3: Vergleich von woody cover-Trajektorien zwischen Feuer- und Nicht-Feuer-Pixeln

# =============================================================================


In [1]:

# ---

# ## 1. Setup & Daten laden

# ---

import numpy as np  
import pandas as pd  
import matplotlib.pyplot as plt  
import seaborn as sns  
import rasterio  
import os  
import time  
from tqdm import tqdm  
from scipy import stats  
from sklearn.preprocessing import StandardScaler  
from sklearn.neighbors import NearestNeighbors  
from sklearn_extra.cluster import KMedoids  
from sklearn.ensemble import RandomForestClassifier  
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score  
from scipy.stats import kruskal, mannwhitneyu


In [2]:

# === Pfade ===

_workDir_remote = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
_workDir_local  = r"D:\Seafile\Meine Bibliothek\uni\master\thesis"
workDir = _workDir_local  
combined_path = os.path.join(workDir, r"_Runs/05_Landcover_integrated/woody_burned_MBA_MODIS_IGBP_combined.tif")  
ecoregion_raster_path = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_500m_3035_MBA.tif")
ecoregion_attr_path   = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv")
out_dir = os.path.join(workDir, "_Runs", "07_Fire_non_Fire_comparison")  
os.makedirs(out_dir, exist_ok=True)

# === Band-Konfiguration ===

years_woody = list(range(1985, 2025))  
years_burned = list(range(2000, 2026))  
modis_years = list(range(2001, 2025))

WOODY_START = 1  
MBA_START = len(years_woody) + 1  
MODIS_START = len(years_woody) + len(years_burned) * 12 + 1

# === Lade Daten ===

print("Lade Woody Cover...")  
with rasterio.open(combined_path) as src:  
    woody_bands = list(range(WOODY_START, WOODY_START + len(years_woody)))  
    woody = src.read(woody_bands)  
    nodata = src.nodata  
    transform = src.transform  
    crs = src.crs  
    height, width = src.height, src.width  
    meta = src.meta.copy()

print("Lade Burned Area...")  
with rasterio.open(combined_path) as src:  
    burned_band_indices = [MBA_START + (i * 12) for i in range(len(years_burned))]  
    burned = src.read(burned_band_indices)

print("Lade MODIS IGBP...")  
with rasterio.open(combined_path) as src:  
    modis_bands = list(range(MODIS_START, MODIS_START + len(modis_years)))  
    modis_igbp = src.read(modis_bands)

print("Lade Ecoregion-Raster...")  
with rasterio.open(ecoregion_raster_path) as src:  
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)  
eco_mapping = {}  
for _, row in ecoregion_df.iterrows():  
    eco_mapping[row['NUMERIC_ID']] = {  
        'code': row['code'], 'name': row['name'], 'hex_color': row['hex_color']  
    }


Lade Woody Cover...
Lade Burned Area...
Lade MODIS IGBP...
Lade Ecoregion-Raster...


In [ ]:

# === Vorberechnungen ===

fire_counts = np.sum(burned == 1, axis=0)  
first_fire_idx = np.argmax(burned == 1, axis=0)

print("\n✓ Alle Daten geladen!")

# ---

# ## 2. Mediterran-Maske & Pixel-Kategorisierung

# ---

print("\n2. MEDITERRAN-MASKE & PIXEL-KATEGORISIERUNG")  
print("-" * 70)

# Finde Mediterranean Ecoregion ID

MED_ID = None  
for eco_id, info in eco_mapping.items():  
    if info['code'] == 'MED':  
        MED_ID = eco_id  
        break

if MED_ID is None:  
    for eco_id, info in eco_mapping.items():  
        if 'mediterr' in info['name'].lower():  
            MED_ID = eco_id  
            break

print(f"  Mediterranean ID: {MED_ID}")  
print(f"  Name: {eco_mapping[MED_ID]['name']}")

# Erstelle Maske: Mediterran

med_mask = (eco_raster == MED_ID)  
n_med_total = np.sum(med_mask)

# Pixel mit Feuer vs. ohne Feuer

med_with_fire_mask = med_mask & (fire_counts > 0)  
med_no_fire_mask = med_mask & (fire_counts == 0)

n_med_with_fire = np.sum(med_with_fire_mask)  
n_med_no_fire = np.sum(med_no_fire_mask)

print(f"\n  Pixel in Mediterran gesamt: {n_med_total:,}")  
print(f"  Pixel mit ≥1 Brand:         {n_med_with_fire:,}")  
print(f"  Pixel ohne Brand:           {n_med_no_fire:,}")

# ---

# ## 3. Feature-Extraktion für Feuer- und Nicht-Feuer-Pixel

# ---

print("\n3. FEATURE-EXTRAKTION")  
print("-" * 70)

# === Pre-Fire Landcover (wie in 05_landcover_analysis) ===

MAJOR_LC_NAMES = ['', 'Forests', 'Shrublands', 'Savannas', 'Grasslands',  
                  'Wetlands', 'Croplands', 'Urban', 'Barren/Ice', 'Water', 'Other']  
igbp_to_major_id = np.array([0, 1, 1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 6, 8, 8, 9], dtype=np.uint8)

# === Feuer-Pixel: Extrahiere Features ===

y_fire, x_fire = np.where(med_with_fire_mask)  
n_fire = len(y_fire)  
print(f"  Extrahiere Features für {n_fire:,} Feuer-Pixel...")

offset = years_burned[0] - years_woody[0]  
fire_idx_fire = first_fire_idx[y_fire, x_fire]

# Relative Jahre: -5 bis +10

rel_years_range = list(range(-5, 11))  
pixel_trajectories_fire = np.full((n_fire, len(rel_years_range)), np.nan, dtype=np.float32)

print("  Extrahiere Trajektorien für Feuer-Pixel...")  
for j, rel_year in enumerate(tqdm(rel_years_range, desc="Rel Years")):  
    woody_band_idx = fire_idx_fire + rel_year + offset - 1  
    valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))  
    if np.sum(valid) > 0:  
        vals = woody[woody_band_idx[valid], y_fire[valid], x_fire[valid]].astype(np.float32)  
        vals[vals == nodata] = np.nan  
        pixel_trajectories_fire[valid, j] = vals

# Berechne 8 Features für Feuer-Pixel

pre_fire_fire = pixel_trajectories_fire[:, 0:5]  
fire_year_fire = pixel_trajectories_fire[:, 5]  
year_m1_fire = pixel_trajectories_fire[:, 4]  
year_p1_fire = pixel_trajectories_fire[:, 6]  
year_p3_fire = pixel_trajectories_fire[:, 8]  
year_p5_fire = pixel_trajectories_fire[:, 10]

feat_pre_fire_mean_fire = np.nanmean(pre_fire_fire, axis=1)  
feat_pre_fire_trend_fire = np.full(n_fire, np.nan, dtype=np.float32)  
x_trend = np.arange(5, dtype=np.float32)  
for i in range(n_fire):  
    vals_i = pre_fire_fire[i, :]  
    valid_i = ~np.isnan(vals_i)  
    if np.sum(valid_i) >= 3:  
        slope, _, _, _, _ = stats.linregress(x_trend[valid_i], vals_i[valid_i])  
        feat_pre_fire_trend_fire[i] = slope

feat_fire_loss_fire = year_m1_fire - fire_year_fire  
feat_fire_loss_pct_fire = np.where(year_m1_fire > 0, feat_fire_loss_fire / year_m1_fire * 100, np.nan)  
feat_recovery_1yr_fire = year_p1_fire - fire_year_fire  
feat_recovery_3yr_fire = year_p3_fire - fire_year_fire  
feat_recovery_5yr_fire = year_p5_fire - fire_year_fire  
feat_recovery_rate_5yr_fire = np.where(  
    feat_fire_loss_fire > 0,  
    feat_recovery_5yr_fire / feat_fire_loss_fire * 100,  
    np.nan  
)

# === Nicht-Feuer-Pixel: Extrahiere Features ===

y_no_fire, x_no_fire = np.where(med_no_fire_mask)  
n_no_fire = len(y_no_fire)  
print(f"  Extrahiere Features für {n_no_fire:,} Nicht-Feuer-Pixel...")

# Relative Jahre: -5 bis +10 (als "Pre-Fire"-Äquivalent)

pixel_trajectories_no_fire = np.full((n_no_fire, len(rel_years_range)), np.nan, dtype=np.float32)

print("  Extrahiere Trajektorien für Nicht-Feuer-Pixel...")  
for j, rel_year in enumerate(tqdm(rel_years_range, desc="Rel Years")):  
    woody_band_idx = rel_year + offset - 1  # Kein Offset für Nicht-Feuer!  
    valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))  
    if np.sum(valid) > 0:  
        vals = woody[woody_band_idx, y_no_fire[valid], x_no_fire[valid]].astype(np.float32)  
        vals[vals == nodata] = np.nan  
        pixel_trajectories_no_fire[valid, j] = vals

# Berechne Features für Nicht-Feuer-Pixel (nur Pre-Fire-ähnliche Features)

pre_fire_no_fire = pixel_trajectories_no_fire[:, 0:5]  
feat_pre_fire_mean_no_fire = np.nanmean(pre_fire_no_fire, axis=1)  
feat_pre_fire_trend_no_fire = np.full(n_no_fire, np.nan, dtype=np.float32)  
for i in range(n_no_fire):  
    vals_i = pre_fire_no_fire[i, :]  
    valid_i = ~np.isnan(vals_i)  
    if np.sum(valid_i) >= 3:  
        slope, _, _, _, _ = stats.linregress(x_trend[valid_i], vals_i[valid_i])  
        feat_pre_fire_trend_no_fire[i] = slope

# === Kombiniere Features ===

FEATURE_NAMES = [  
    'pre_fire_mean', 'pre_fire_trend', 'fire_loss', 'fire_loss_pct',  
    'recovery_1yr', 'recovery_3yr', 'recovery_5yr', 'recovery_rate_5yr'  
]

feature_matrix_fire = np.column_stack([  
    feat_pre_fire_mean_fire, feat_pre_fire_trend_fire, feat_fire_loss_fire,  
    feat_fire_loss_pct_fire, feat_recovery_1yr_fire, feat_recovery_3yr_fire,  
    feat_recovery_5yr_fire, feat_recovery_rate_5yr_fire  
])

feature_matrix_no_fire = np.column_stack([  
    feat_pre_fire_mean_no_fire, feat_pre_fire_trend_no_fire,  
    np.full(n_no_fire, np.nan), np.full(n_no_fire, np.nan),  
    np.full(n_no_fire, np.nan), np.full(n_no_fire, np.nan),  
    np.full(n_no_fire, np.nan), np.full(n_no_fire, np.nan)  
])

# === Entferne Pixel mit NaN ===

nan_count_fire = np.isnan(feature_matrix_fire).sum(axis=1)  
valid_fire = nan_count_fire == 0  
feature_matrix_fire_clean = feature_matrix_fire[valid_fire]  
y_fire_valid = y_fire[valid_fire]  
x_fire_valid = x_fire[valid_fire]  
trajectories_fire_clean = pixel_trajectories_fire[valid_fire]

nan_count_no_fire = np.isnan(feature_matrix_no_fire).sum(axis=1)  
valid_no_fire = nan_count_no_fire == 0  
feature_matrix_no_fire_clean = feature_matrix_no_fire[valid_no_fire]  
y_no_fire_valid = y_no_fire[valid_no_fire]  
x_no_fire_valid = x_no_fire[valid_no_fire]  
trajectories_no_fire_clean = pixel_trajectories_no_fire[valid_no_fire]

n_valid_fire = feature_matrix_fire_clean.shape[0]  
n_valid_no_fire = feature_matrix_no_fire_clean.shape[0]

print(f"\n  ✓ Gültige Feuer-Pixel: {n_valid_fire:,} ({n_valid_fire/n_fire*100:.1f}%)")*  
*print(f"  ✓ Gültige Nicht-Feuer-Pixel: {n_valid_no_fire:,} ({n_valid_no_fire/n_no_fire*100:.1f}%)")

# ---

# ## 4. Covariate Matching (Feuer ↔ Nicht-Feuer)

# ---

print("\n4. COVARIATE MATCHING")  
print("-" * 70)

# === Matching-Kriterien ===

# Nutze Pre-Fire Woody Cover und LULC als Matching-Variablen

X_fire = feature_matrix_fire_clean[:, [0, 1]]  # pre_fire_mean, pre_fire_trend  
X_no_fire = feature_matrix_no_fire_clean[:, [0, 1]]

# === Matching ===

print("  Finde passende Nicht-Feuer-Pixel für Feuer-Pixel...")  
nbrs = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(X_no_fire)  
_, indices = nbrs.kneighbors(X_fire)  
matched_no_fire_indices = indices.flatten()

# === Extrahiere gematchte Nicht-Feuer-Pixel ===

feature_matrix_matched = feature_matrix_no_fire_clean[matched_no_fire_indices]  
trajectories_matched = trajectories_no_fire_clean[matched_no_fire_indices]  
y_matched = y_no_fire_valid[matched_no_fire_indices]  
x_matched = x_no_fire_valid[matched_no_fire_indices]

print(f"  ✓ {n_valid_fire} Feuer-Pixel wurden mit {n_valid_no_fire} Nicht-Feuer-Pixeln gematcht.")

# ---

# ## 5. Clustering (k-Medoids) auf Feuer + Nicht-Feuer

# ---

print("\n5. CLUSTERING (k-MEDOIDS)")  
print("-" * 70)

# === Kombiniere Features ===

feature_matrix_combined = np.vstack([  
    feature_matrix_fire_clean,  
    feature_matrix_matched  
])  
n_combined = feature_matrix_combined.shape[0]

# === Standardisierung ===

scaler = StandardScaler()  
feature_matrix_scaled = scaler.fit_transform(feature_matrix_combined)

# === SOM-Training ===

from minisom import MiniSom  
som_side = min(25, int(np.ceil(5 * np.sqrt(n_combined) ** 0.5)))  
som_side = max(10, min(som_side, 25))  
print(f"  SOM-Größe: {som_side} × {som_side} = {som_side**2} Neuronen")

som = MiniSom(  
    som_side, som_side,  
    feature_matrix_scaled.shape[1],  
    sigma=max(1.0, som_side / 4),  
    learning_rate=0.5,  
    neighborhood_function='gaussian',  
    random_seed=42  
)  
print("  Initialisiere mit PCA...")  
som.pca_weights_init(feature_matrix_scaled)

n_iterations = 50000  
print(f"  Trainiere SOM ({n_iterations:,} Iterationen)...")  
t0 = time.time()  
som.train_random(feature_matrix_scaled, n_iterations, verbose=False)  
print(f"  ✓ SOM-Training abgeschlossen ({time.time()-t0:.1f}s)")

# === Mapping: Pixel → SOM-Knoten ===

print("  Weise alle Pixel SOM-Knoten zu...")  
t0 = time.time()  
bmu_indices = np.array([som.winner(x) for x in tqdm(feature_matrix_scaled, desc="BMU")])  
print(f"  ✓ BMU-Zuweisung abgeschlossen ({time.time()-t0:.1f}s)")

# === k-Medoids Clustering ===

K_RANGE = range(2, 13)  
validation_results = []

print("  Teste k-Werte...")  
for k in tqdm(K_RANGE, desc="k-Medoids"):  
    kmed = KMedoids(n_clusters=k, metric='euclidean', random_state=42, max_iter=300)  
    codebook = som.get_weights().reshape(-1, feature_matrix_scaled.shape[1])  
    codebook_labels = kmed.fit_predict(codebook)

```
bmu_flat = bmu_indices[:, 0] * som_side + bmu_indices[:, 1]
pixel_labels = codebook_labels[bmu_flat]

# Subsample für Metrik-Berechnung
if n_combined > 50000:
    rng = np.random.default_rng(42)
    eval_idx = rng.choice(n_combined, 50000, replace=False)
else:
    eval_idx = np.arange(n_combined)

sil = silhouette_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx],
                       sample_size=min(10000, len(eval_idx)))
ch = calinski_harabasz_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx])
db = davies_bouldin_score(feature_matrix_scaled[eval_idx], pixel_labels[eval_idx])

validation_results.append({
    'k': k,
    'silhouette': sil,
    'calinski_harabasz': ch,
    'davies_bouldin': db,
    'inertia': kmed.inertia_
})
```

df_validation = pd.DataFrame(validation_results)

# === Visualisierung der Validierungsmetriken ===

fig, axes = plt.subplots(2, 2, figsize=(14, 10))  
ax = axes[0, 0]  
ax.plot(df_validation['k'], df_validation['silhouette'], 'bo-', linewidth=2)  
ax.set_xlabel('k')  
ax.set_ylabel('Silhouette Score')  
ax.set_title('Silhouette Score (higher = better)')  
ax.grid(True, alpha=0.3)  
best_sil_k = df_validation.loc[df_validation['silhouette'].idxmax(), 'k']  
ax.axvline(x=best_sil_k, color='red', linestyle='--', alpha=0.7, label=f'Best: k={best_sil_k}')  
ax.legend()

ax = axes[0, 1]  
ax.plot(df_validation['k'], df_validation['calinski_harabasz'], 'go-', linewidth=2)  
ax.set_xlabel('k')  
ax.set_ylabel('Calinski-Harabasz Index')  
ax.set_title('Calinski-Harabasz (higher = better)')  
ax.grid(True, alpha=0.3)

ax = axes[1, 0]  
ax.plot(df_validation['k'], df_validation['davies_bouldin'], 'ro-', linewidth=2)  
ax.set_xlabel('k')  
ax.set_ylabel('Davies-Bouldin Index')  
ax.set_title('Davies-Bouldin (lower = better)')  
ax.grid(True, alpha=0.3)  
best_db_k = df_validation.loc[df_validation['davies_bouldin'].idxmin(), 'k']  
ax.axvline(x=best_db_k, color='blue', linestyle='--', alpha=0.7, label=f'Best: k={best_db_k}')  
ax.legend()

ax = axes[1, 1]  
ax.plot(df_validation['k'], df_validation['inertia'], 'mo-', linewidth=2)  
ax.set_xlabel('k')  
ax.set_ylabel('Inertia')  
ax.set_title('Inertia / Elbow (lower = better)')  
ax.grid(True, alpha=0.3)

plt.suptitle('Cluster Validation Metrics – Feuer vs. Nicht-Feuer', fontsize=14, fontweight='bold')  
plt.tight_layout()  
val_plot_path = os.path.join(out_dir, "cluster_validation_metrics.png")  
plt.savefig(val_plot_path, dpi=300, bbox_inches='tight')  
plt.close()  
print(f"  ✓ {val_plot_path}")

# Speichere Validierungsergebnisse

df_validation.to_csv(os.path.join(out_dir, "cluster_validation_results.csv"), index=False)

print(f"\n  Empfohlene k-Werte:")  
print(f"    Best Silhouette:     k = {best_sil_k}")  
print(f"    Best Davies-Bouldin: k = {best_db_k}")  
print(f"\n  ⚠️  Wähle k im nächsten Schritt manuell basierend auf den Metriken!")

# ---

# ## 6. Finale Clustering mit gewähltem k

# ---

print("\n6. FINALE CLUSTERUNG")  
print("-" * 70)

CHOSEN_K = 5  # <-- ANPASSEN nach Schritt 5!  
print(f"  Gewähltes k = {CHOSEN_K}")

kmed_final = KMedoids(n_clusters=CHOSEN_K, metric='euclidean', random_state=42, max_iter=300)  
codebook = som.get_weights().reshape(-1, feature_matrix_scaled.shape[1])  
codebook_labels_final = kmed_final.fit_predict(codebook)

bmu_flat = bmu_indices[:, 0] * som_side + bmu_indices[:, 1]  
cluster_labels = codebook_labels_final[bmu_flat]

# === Cluster-Größen ===

unique_clusters, cluster_counts = np.unique(cluster_labels, return_counts=True)  
print(f"\n  Cluster-Verteilung:")  
for cl, cnt in zip(unique_clusters, cluster_counts):  
    print(f"    Cluster {cl}: {cnt:,} Pixel ({cnt/n_combined*100:.1f}%)")

# === Label Feuer vs. Nicht-Feuer ===

fire_labels = np.concatenate([  
    np.ones(n_valid_fire, dtype=int),  # Feuer-Pixel = Label 1  
    np.zeros(n_valid_no_fire, dtype=int)  # Nicht-Feuer-Pixel = Label 0  
])

# ---

# ## 7. Cluster-Interpretation

# ---

print("\n7. CLUSTER-INTERPRETATION")  
print("-" * 70)

# === 7a: Mittlere Trajektorien pro Cluster ===

CLUSTER_COLORS = plt.cm.Set1(np.linspace(0, 1, CHOSEN_K))  
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Plot A: Alle Cluster zusammen

ax = axes[0]  
for cl in range(CHOSEN_K):  
    mask_cl = cluster_labels == cl  
    mean_traj = np.nanmean(np.vstack([trajectories_fire_clean, trajectories_matched])[mask_cl], axis=0)  
    std_traj = np.nanstd(np.vstack([trajectories_fire_clean, trajectories_matched])[mask_cl], axis=0)

```
ax.plot(rel_years_range, mean_traj, marker='o', linewidth=2.5,
        color=CLUSTER_COLORS[cl], label=f'Cluster {cl} (n={np.sum(mask_cl):,})')

lower = np.maximum(mean_traj - std_traj, 0)
upper = np.minimum(mean_traj + std_traj, 100)
ax.fill_between(rel_years_range, lower, upper, alpha=0.15, color=CLUSTER_COLORS[cl])
```

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)  
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)  
ax.set_xlabel('Years Relative to Fire', fontsize=12)  
ax.set_ylabel('Woody Cover (%)', fontsize=12)  
ax.set_title('Mean Trajectories per Cluster', fontsize=13, fontweight='bold')  
ax.legend(loc='best', fontsize=10)  
ax.grid(True, alpha=0.3)  
ax.set_xlim(-5.5, 10.5)

# Plot B: Feature-Vergleich als Heatmap

ax = axes[1]  
ax.axis('off')

cluster_feature_means = np.zeros((CHOSEN_K, len(FEATURE_NAMES)))  
for cl in range(CHOSEN_K):  
    mask_cl = cluster_labels == cl  
    cluster_feature_means[cl] = np.nanmean(feature_matrix_combined[mask_cl], axis=0)

df_cluster_features = pd.DataFrame(  
    cluster_feature_means,  
    index=[f'Cluster {cl}' for cl in range(CHOSEN_K)],  
    columns=FEATURE_NAMES  
)

ax2 = fig.add_subplot(122)  
sns.heatmap(df_cluster_features, cmap='RdYlGn', annot=True, fmt='.1f',  
            ax=ax2, linewidths=0.5, center=0,  
            cbar_kws={'label': 'Mean Feature Value (unstandardized)'})  
ax2.set_title('Cluster Feature Profiles', fontsize=13, fontweight='bold')  
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle(f'Mediterranean Trajectory Clusters (k={CHOSEN_K})', fontsize=15, fontweight='bold')  
plt.tight_layout()  
traj_plot_path = os.path.join(out_dir, "cluster_trajectories.png")  
plt.savefig(traj_plot_path, dpi=300, bbox_inches='tight')  
plt.close()  
print(f"  ✓ {traj_plot_path}")

# === 7b: Vergleich Feuer vs. Nicht-Feuer pro Cluster ===

print("\n7b. Vergleich Feuer vs. Nicht-Feuer pro Cluster...")

for cl in range(CHOSEN_K):  
    mask_cl = cluster_labels == cl  
    fire_in_cluster = np.sum(fire_labels[mask_cl] == 1)  
    no_fire_in_cluster = np.sum(fire_labels[mask_cl] == 0)

```
print(f"\n  Cluster {cl}:")
print(f"    Feuer-Pixel: {fire_in_cluster:,} ({fire_in_cluster/np.sum(mask_cl)*100:.1f}%)")
print(f"    Nicht-Feuer-Pixel: {no_fire_in_cluster:,} ({no_fire_in_cluster/np.sum(mask_cl)*100:.1f}%)")

# Statistischer Vergleich der Erholungsraten
if fire_in_cluster > 0 and no_fire_in_cluster > 0:
    recovery_fire = feature_matrix_combined[mask_cl & (fire_labels == 1), 7]  # recovery_rate_5yr
    recovery_no_fire = feature_matrix_combined[mask_cl & (fire_labels == 0), 7]

    stat, p = mannwhitneyu(recovery_fire, recovery_no_fire, alternative='two-sided')
    print(f"    Erholungsrate (5Jahre): Feuer vs. Nicht-Feuer")
    print(f"      Feuer-Median: {np.nanmedian(recovery_fire):.1f}%")
    print(f"      Nicht-Feuer-Median: {np.nanmedian(recovery_no_fire):.1f}%")
    print(f"      p-Wert (Mann-Whitney): {p:.4f}")
```

# ---

# ## 8. Modellierung: Clusterzugehörigkeit erklären

# ---

print("\n8. MODELLIERUNG: CLUSTERZUGEHÖRIGKEIT ERKLÄREN")  
print("-" * 70)

# === Zielvariable: Clusterzugehörigkeit ===

y_cluster = cluster_labels

# === Prädiktoren ===

# Feuer-Status (1=Feuer, 0=Nicht-Feuer)

X_features = np.column_stack([  
    fire_labels,  # Feuer vs. Nicht-Feuer  
    feature_matrix_combined[:, 0],  # pre_fire_mean  
    feature_matrix_combined[:, 1],  # pre_fire_trend  
])

# === Modell: Random Forest ===

print("  Trainiere Random Forest...")  
rf = RandomForestClassifier(n_estimators=100, random_state=42)  
rf.fit(X_features, y_cluster)

# === Feature Importance ===

feature_names = ['Feuer (1=ja)', 'Pre-Fire Woody Cover', 'Pre-Fire Trend']  
importances = rf.feature_importances_  
indices = np.argsort(importances)[::-1]

print("\n  Feature Importance:")  
for f in range(X_features.shape[1]):  
    print(f"    {feature_names[indices[f]]}: {importances[indices[f]]:.3f}")

# === Visualisierung ===

plt.figure(figsize=(10, 6))  
plt.title("Feature Importance – Clusterzugehörigkeit")  
plt.bar(range(X_features.shape[1]), importances[indices], align="center")  
plt.xticks(range(X_features.shape[1]), [feature_names[i] for i in indices], rotation=45)  
plt.ylabel("Importance")  
plt.tight_layout()  
importance_plot_path = os.path.join(out_dir, "feature_importance.png")  
plt.savefig(importance_plot_path, dpi=300, bbox_inches='tight')  
plt.close()  
print(f"  ✓ {importance_plot_path}")

# ---

# ## 9. Export der Ergebnisse

# ---

print("\n9. EXPORT DER ERGEBNISSE")  
print("-" * 70)

# === Speichere Cluster-Labels als Raster ===

cluster_raster = np.full((height, width), -1, dtype=np.int16)  
cluster_raster[y_fire_valid, x_fire_valid] = cluster_labels[:n_valid_fire]  
cluster_raster[y_matched, x_matched] = cluster_labels[n_valid_fire:]

meta.update(dtype='int16', nodata=-1)  
with rasterio.open(  
    os.path.join(out_dir, "cluster_labels_mediterranean.tif"),  
    'w', **meta  
) as dst:  
    dst.write(cluster_raster, 1)

print(f"  ✓ Cluster-Labels als Raster gespeichert: cluster_labels_mediterranean.tif")

# === Speichere Validierungsmetriken ===

df_validation.to_csv(os.path.join(out_dir, "cluster_validation_results.csv"), index=False)  
print(f"  ✓ Validierungsmetriken gespeichert: cluster_validation_results.csv")

print("\n✅ ANALYSE ABGESCHLOSSEN!")  
print(f"   Ergebnisse wurden in {out_dir} gespeichert.")